# Data Pre-processing of NSW Electricity Demand

## Section 1 - Objective

The purpose of this notebook is to prepare the NSW electricity demand dataset for forecasting models.

This includes:
1. converting the datetime field into a proper time index,
2. checking missing values,
3. confirming alignment across datasets,
4. selecting modelling variables,
5. splitting the data into training and testing sets.

## Section 2 - Import Libraries and Load Raw Data

In [ ]:
#!git clone https://github.com/laukamkit/capstone_project_GroupA.git

fatal: destination path 'capstone_project_GroupA' already exists and is not an empty directory.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

repo_path = os.path.join(os.path.dirname(__file__), "..")
nsw_path = os.path.join(repo_path, "data", "NSW")

part_a = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip.partaa")
part_b = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip.partab")
forecast_zip = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip")

if not os.path.exists(forecast_zip):
    with open(forecast_zip, "wb") as outfile:
        for p in [part_a, part_b]:
            with open(p, "rb") as infile:
                outfile.write(infile.read())

df_demand = pd.read_csv(os.path.join(nsw_path, "totaldemand_nsw.csv.zip"))
df_temp = pd.read_csv(os.path.join(nsw_path, "temperature_nsw.csv.zip"))
df_forecast = pd.read_csv(forecast_zip)

print("Demand shape:", df_demand.shape)
print("Temp shape:", df_temp.shape)
print("Forecast shape:", df_forecast.shape)


KeyboardInterrupt



## Section 3. Convert Datetime Format

Convert "LASTCHANGED" in table df_forecast to datetime and round it so it can align with the demand timestamps.

In [3]:
df_demand["DATETIME"] = pd.to_datetime(df_demand["DATETIME"], dayfirst=True)
df_temp["DATETIME"] = pd.to_datetime(df_temp["DATETIME"], dayfirst=True)
df_forecast["LASTCHANGED"] = pd.to_datetime(df_forecast["LASTCHANGED"])
df_forecast["DATETIME_ROUND"] = df_forecast["LASTCHANGED"].dt.round("30min")
print(df_demand.dtypes)
print(df_temp.dtypes)
print(df_forecast.dtypes)
df_forecast[["LASTCHANGED","DATETIME_ROUND"]].head()

DATETIME       datetime64[us]
TOTALDEMAND           float64
REGIONID                  str
dtype: object
LOCATION                  str
DATETIME       datetime64[us]
TEMPERATURE           float64
dtype: object
PREDISPATCHSEQNO             int64
REGIONID                       str
PERIODID                     int64
FORECASTDEMAND             float64
LASTCHANGED         datetime64[us]
DATETIME                       str
DATETIME_ROUND      datetime64[us]
dtype: object


,LASTCHANGED,DATETIME_ROUND
0,2009-12-30 12:31:49,2009-12-30 12:30:00
1,2009-12-30 13:01:43,2009-12-30 13:00:00
2,2009-12-30 13:31:36,2009-12-30 13:30:00
3,2009-12-30 14:01:44,2009-12-30 14:00:00
4,2009-12-30 14:31:35,2009-12-30 14:30:00


Similarly, for df_temp, there are odd minutes in the datetime. Round it to 30 minute intervals.

In [4]:
df_temp["DATETIME_ROUND"] = df_temp["DATETIME"].dt.round("30min")

In [5]:
print(df_forecast.head())
print('\n')
print(df_temp.head())

   PREDISPATCHSEQNO REGIONID  PERIODID  FORECASTDEMAND         LASTCHANGED  \
0        2009123018     NSW1        71         7832.04 2009-12-30 12:31:49   
1        2009123019     NSW1        70         7832.04 2009-12-30 13:01:43   
2        2009123020     NSW1        69         7832.03 2009-12-30 13:31:36   
3        2009123021     NSW1        68         7832.03 2009-12-30 14:01:44   
4        2009123022     NSW1        67         7830.96 2009-12-30 14:31:35   

              DATETIME      DATETIME_ROUND  
0  2010-01-01 00:00:00 2009-12-30 12:30:00  
1  2010-01-01 00:00:00 2009-12-30 13:00:00  
2  2010-01-01 00:00:00 2009-12-30 13:30:00  
3  2010-01-01 00:00:00 2009-12-30 14:00:00  
4  2010-01-01 00:00:00 2009-12-30 14:30:00  


    LOCATION            DATETIME  TEMPERATURE      DATETIME_ROUND
0  Bankstown 2010-01-01 00:00:00         23.1 2010-01-01 00:00:00
1  Bankstown 2010-01-01 00:01:00         23.1 2010-01-01 00:00:00
2  Bankstown 2010-01-01 00:30:00         22.9 2010-01-01 00:3

## Section 4. Aggregate Temperature & Forecast Data by 30-Minute Interval

There may be multiple forecast/temperature rows within the same rounded timestamp, so aggregate them using mean.

In [ ]:
df_forecast_30min = (
    df_forecast
    .groupby("DATETIME_ROUND")["FORECASTDEMAND"]
    .mean()
    .reset_index()
)
df_forecast_30min = df_forecast_30min.rename(
    columns={"DATETIME_ROUND":"DATETIME"}
)

In [7]:
df_temp_30min = df_temp.groupby(["LOCATION","DATETIME_ROUND"])["TEMPERATURE"].mean().reset_index()
df_temp_30min = df_temp_30min.rename(
    columns={"DATETIME_ROUND":"DATETIME"}
)

In [8]:
print(df_temp.count())
print(df_temp_30min.count())

LOCATION          220326
DATETIME          220326
TEMPERATURE       220326
DATETIME_ROUND    220326
dtype: int64
LOCATION       195949
DATETIME       195949
TEMPERATURE    195949
dtype: int64


In [9]:
print(df_forecast_30min.head())
print('\n')
print(df_temp_30min.head())

             DATETIME  FORECASTDEMAND
0 2009-12-30 12:30:00     6913.120000
1 2009-12-30 13:00:00     6912.933333
2 2009-12-30 13:30:00     6912.808889
3 2009-12-30 14:00:00     6912.982222
4 2009-12-30 14:30:00     6912.706667


    LOCATION            DATETIME  TEMPERATURE
0  Bankstown 2010-01-01 00:00:00        23.10
1  Bankstown 2010-01-01 00:30:00        22.90
2  Bankstown 2010-01-01 01:00:00        22.65
3  Bankstown 2010-01-01 01:30:00        22.50
4  Bankstown 2010-01-01 02:00:00        22.50


### Also get the best forecast in terms of MSE.

In [67]:
df_best_forecast_30min = (
    df_forecast
    .groupby(["DATETIME_ROUND","PERIODID"])["FORECASTDEMAND"]
    .mean()
    .reset_index()
)
df_best_forecast_30min = df_best_forecast_30min.rename(
    columns={"DATETIME_ROUND":"DATETIME"}
)

In [70]:
# pivot to have separate columns for each period's forecast
df_best_forecast_30min = df_best_forecast_30min.pivot(
    index="DATETIME",
    columns="PERIODID",
    values="FORECASTDEMAND"
).reset_index()

PERIODID,DATETIME,1,2,3,4,5,6,7,8,9,...,70,71,72,73,74,75,76,77,78,79
0,2009-12-30 12:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,7832.04,7725.11,7487.61,7136.08,6801.07,6547.82,6352.30,6193.37,6142.68
1,2009-12-30 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7832.04,7723.11,7487.63,7136.31,6801.06,6547.93,6351.37,6193.42,6143.53,NaN
2,2009-12-30 13:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7724.10,7487.59,7134.93,6801.22,6547.85,6352.12,6193.16,6142.28,NaN,NaN
3,2009-12-30 14:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7489.41,7134.87,6801.24,6546.87,6352.04,6193.00,6143.28,NaN,NaN,NaN
4,2009-12-30 14:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7134.53,6801.26,6547.10,6352.36,6193.27,6143.45,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196565,2021-03-17 21:30:00,7409.33,7379.27,7283.91,7133.90,7028.45,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196566,2021-03-17 22:00:00,7422.63,7316.62,7154.65,7041.65,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196567,2021-03-17 22:30:00,7313.13,7187.72,7087.60,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196568,2021-03-17 23:00:00,7192.94,7087.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Section 5. Merge the Three Datasets

Merge using DATETIME as the key.

In [144]:
df_merged = (
    df_demand
    .merge(df_temp_30min, on="DATETIME", how="inner")
    .merge(df_forecast_30min, on="DATETIME", how="inner")
    .merge(df_best_forecast_30min, on="DATETIME", how="inner")
)

Fill each forecast periods' NaN values with the average of the forecasts.

In [145]:
df_merged.iloc[:, 6:] = df_merged.apply(lambda x: x[6:].fillna(x['FORECASTDEMAND']), axis=1)

In [146]:
df_merged.head()

,DATETIME,TOTALDEMAND,REGIONID,LOCATION,TEMPERATURE,FORECASTDEMAND,1,2,3,4,...,70,71,72,73,74,75,76,77,78,79
0,2010-01-01 00:00:00,8038.00,NSW1,Bankstown,23.10,7544.100179,7596.21,7304.27,6976.87,6655.83,...,7544.100179,7544.100179,7544.100179,7544.100179,7544.100179,7544.100179,7544.100179,7544.100179,7544.100179,7544.100179
1,2010-01-01 00:30:00,7809.31,NSW1,Bankstown,22.90,7433.778364,7380.70,7046.39,6714.37,6422.37,...,7433.778364,7433.778364,7433.778364,7433.778364,7433.778364,7433.778364,7433.778364,7433.778364,7433.778364,7433.778364
2,2010-01-01 01:00:00,7483.69,NSW1,Bankstown,22.65,7427.407037,7022.05,6680.15,6388.86,6217.14,...,7427.407037,7427.407037,7427.407037,7427.407037,7427.407037,7427.407037,7427.407037,7427.407037,7427.407037,7427.407037
3,2010-01-01 01:30:00,7117.23,NSW1,Bankstown,22.50,7433.393774,6682.92,6383.92,6210.34,6067.49,...,7433.393774,7433.393774,7433.393774,7433.393774,7433.393774,7433.393774,7433.393774,7433.393774,7433.393774,7433.393774
4,2010-01-01 02:00:00,6812.03,NSW1,Bankstown,22.50,7446.699231,6381.27,6212.84,6063.20,5999.54,...,7446.699231,7446.699231,7446.699231,7446.699231,7446.699231,7446.699231,7446.699231,7446.699231,7446.699231,7446.699231


Get MSE for each forecast period

In [147]:
mse_results = []
for col in df_merged.columns[6:]:
    mse_results.append(((df_merged["TOTALDEMAND"] - df_merged[col]) ** 2).mean())
df_merged = pd.concat([df_merged.iloc[:, 0:6], df_merged.loc[:, df_merged.columns[6:][np.argmin(mse_results)]]], axis=1)
df_merged.rename(columns={df_merged.columns[6]: "BEST_FORECASTDEMAND"}, inplace=True)

In [148]:
df_merged

,DATETIME,TOTALDEMAND,REGIONID,LOCATION,TEMPERATURE,FORECASTDEMAND,BEST_FORECASTDEMAND
0,2010-01-01 00:00:00,8038.00,NSW1,Bankstown,23.100000,7544.100179,7596.21
1,2010-01-01 00:30:00,7809.31,NSW1,Bankstown,22.900000,7433.778364,7380.70
2,2010-01-01 01:00:00,7483.69,NSW1,Bankstown,22.650000,7427.407037,7022.05
3,2010-01-01 01:30:00,7117.23,NSW1,Bankstown,22.500000,7433.393774,6682.92
4,2010-01-01 02:00:00,6812.03,NSW1,Bankstown,22.500000,7446.699231,6381.27
...,...,...,...,...,...,...,...
195930,2021-03-17 21:30:00,7503.12,NSW1,Bankstown,19.700000,7246.972000,7409.33
195931,2021-03-17 22:00:00,7419.77,NSW1,Bankstown,19.700000,7233.887500,7422.63
195932,2021-03-17 22:30:00,7417.91,NSW1,Bankstown,19.500000,7196.150000,7313.13
195933,2021-03-17 23:00:00,7287.32,NSW1,Bankstown,19.100000,7140.110000,7192.94


## Section 6. Sort and Set Time Index

Sorting and setting time index is necessary because time-series models require an ordered time index.

In [149]:
df_merged = df_merged.sort_values("DATETIME")
df_merged = df_merged.set_index("DATETIME")
df_merged.head()

,TOTALDEMAND,REGIONID,LOCATION,TEMPERATURE,FORECASTDEMAND,BEST_FORECASTDEMAND
DATETIME,,,,,,
2010-01-01 00:00:00,8038.00,NSW1,Bankstown,23.10,7544.100179,7596.21
2010-01-01 00:30:00,7809.31,NSW1,Bankstown,22.90,7433.778364,7380.70
2010-01-01 01:00:00,7483.69,NSW1,Bankstown,22.65,7427.407037,7022.05
2010-01-01 01:30:00,7117.23,NSW1,Bankstown,22.50,7433.393774,6682.92
2010-01-01 02:00:00,6812.03,NSW1,Bankstown,22.50,7446.699231,6381.27


## Section 7. Check Missing Values

In [150]:
df_merged.isnull().sum()

TOTALDEMAND            0
REGIONID               0
LOCATION               0
TEMPERATURE            0
FORECASTDEMAND         0
BEST_FORECASTDEMAND    0
dtype: int64

The merged dataset was checked for missing values using df_merged.isnull().sum().
No null values were found across all variables, indicating that the merging and alignment of the datasets were successful.

## Section 8. Remove Unnecessary Columns



In [151]:
df_merged["REGIONID"].unique()

<StringArray>
['NSW1']
Length: 1, dtype: str

The REGIONID variable contained only a single value (NSW1) across the entire dataset and therefore did not provide any predictive information. The column was removed from the dataset prior to model training.

In [152]:
df_merged = df_merged.drop(columns=["REGIONID"])

## Section 9. Reindex to Complete 30-Minute Timeline

The time index was examined to ensure continuity at the 30-minute frequency.
A total of 592 timestamps were missing from the expected sequence.
The dataset was therefore reindexed to a complete 30-minute timeline to ensure correct alignment when generating lag features.

In [153]:
# Section: Check missing time points

# expected full 30-minute timeline
full_index = pd.date_range(
    start=df_merged.index.min(),
    end=df_merged.index.max(),
    freq="30min"
)

# find missing timestamps
missing_timepoints = full_index.difference(df_merged.index)

print("Expected number of time points:", len(full_index))
print("Actual number of time points:", len(df_merged.index))
print("Number of missing time points:", len(missing_timepoints))

# preview first few missing timestamps
print(missing_timepoints[:20])

Expected number of time points: 196512
Actual number of time points: 195935
Number of missing time points: 577
DatetimeIndex(['2010-01-10 04:00:00', '2010-01-11 17:00:00',
               '2010-01-14 13:30:00', '2010-01-15 10:30:00',
               '2010-01-16 10:30:00', '2010-01-19 00:30:00',
               '2010-01-19 10:30:00', '2010-01-20 15:30:00',
               '2010-01-21 18:00:00', '2010-01-23 08:30:00',
               '2010-01-23 18:30:00', '2010-01-24 21:30:00',
               '2010-02-01 19:00:00', '2010-02-03 12:00:00',
               '2010-02-05 03:00:00', '2010-02-05 12:00:00',
               '2010-02-09 10:00:00', '2010-02-10 14:30:00',
               '2010-02-10 20:00:00', '2010-02-14 01:30:00'],
              dtype='datetime64[us]', freq=None)


In [154]:
time_gaps = df_merged.index.to_series().diff().value_counts().sort_index()
print(time_gaps)

DATETIME
0 days 00:30:00    195729
0 days 01:00:00       173
0 days 01:30:00        11
0 days 02:00:00         2
0 days 02:30:00         2
0 days 03:00:00         5
0 days 03:30:00         1
0 days 04:00:00         2
0 days 05:00:00         1
0 days 05:30:00         2
0 days 07:30:00         1
0 days 10:00:00         1
0 days 11:30:00         1
0 days 13:00:00         1
0 days 17:30:00         1
3 days 18:30:00         1
Name: count, dtype: int64


In [155]:
# Create full 30-minute time index
full_index = pd.date_range(
    start=df_merged.index.min(),
    end=df_merged.index.max(),
    freq="30min"
)

# Reindex to full timeline
df_merged = df_merged.reindex(full_index)
df_merged.index.name = "DATETIME"

# Check missing values after reindexing
print(df_merged.isnull().sum())

TOTALDEMAND            577
LOCATION               577
TEMPERATURE            577
FORECASTDEMAND         577
BEST_FORECASTDEMAND    577
dtype: int64


Missing numeric values in TOTALDEMAND, TEMPERATURE, and FORECASTDEMAND were filled using time-based interpolation. This method estimates missing observations using surrounding values while preserving the temporal structure of the data.

In [156]:
numeric_cols = ["TOTALDEMAND", "TEMPERATURE", "FORECASTDEMAND"]

df_merged[numeric_cols] = df_merged[numeric_cols].interpolate(method="time")

print(df_merged[numeric_cols].isnull().sum())

TOTALDEMAND       0
TEMPERATURE       0
FORECASTDEMAND    0
dtype: int64


## Section 10. Add Historical Demand Fields

Because the data is every 30 minutes:

1 day ago = 24 hrs x 2 = 48 rows before

1 week ago = 48 x 7 = 336 rows before

1 year ago = 48 x 365 = 17,520 rows before

In [157]:
df_merged["demand_1_day_ago"] = df_merged["TOTALDEMAND"].shift(48)
df_merged["demand_1_week_ago"] = df_merged["TOTALDEMAND"].shift(336)
df_merged["demand_1_year_ago"] = df_merged["TOTALDEMAND"].shift(17520)
df_merged[[
    "TOTALDEMAND",
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
]].head(1000000)

,TOTALDEMAND,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago
DATETIME,,,,
2010-01-01 00:00:00,8038.00,NaN,NaN,NaN
2010-01-01 00:30:00,7809.31,NaN,NaN,NaN
2010-01-01 01:00:00,7483.69,NaN,NaN,NaN
2010-01-01 01:30:00,7117.23,NaN,NaN,NaN
2010-01-01 02:00:00,6812.03,NaN,NaN,NaN
...,...,...,...,...
2021-03-17 21:30:00,7503.12,7462.84,7732.69,7535.95
2021-03-17 22:00:00,7419.77,7373.83,7642.24,7480.90
2021-03-17 22:30:00,7417.91,7345.78,7567.36,7465.50


Rows with NaN entries in demand_1_day_ago, demand_1_week_ago, and demand_1_year_ago are removed to ensure that all lagged demand features are available for model training. These missing values occur at the beginning of the dataset where sufficient historical observations do not exist to compute the lag features.

In [158]:
df_model = df_merged.dropna(subset=[
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
])
df_model[[
    "TOTALDEMAND",
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
]].head(1000000)

,TOTALDEMAND,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago
DATETIME,,,,
2011-01-01 00:00:00,8063.36,7628.57,7358.22,8038.00
2011-01-01 00:30:00,7844.62,7384.33,7141.33,7809.31
2011-01-01 01:00:00,7545.29,7161.66,6870.62,7483.69
2011-01-01 01:30:00,7206.58,6842.79,6557.80,7117.23
2011-01-01 02:00:00,6824.75,6594.82,6237.94,6812.03
...,...,...,...,...
2021-03-17 21:30:00,7503.12,7462.84,7732.69,7535.95
2021-03-17 22:00:00,7419.77,7373.83,7642.24,7480.90
2021-03-17 22:30:00,7417.91,7345.78,7567.36,7465.50


In [159]:
df_model.reset_index(inplace=True)
df_model.head()

,DATETIME,TOTALDEMAND,LOCATION,TEMPERATURE,FORECASTDEMAND,BEST_FORECASTDEMAND,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago
0,2011-01-01 00:00:00,8063.36,Bankstown,21.0,8660.258214,7882.11,7628.57,7358.22,8038.00
1,2011-01-01 00:30:00,7844.62,Bankstown,20.3,8674.982909,7598.89,7384.33,7141.33,7809.31
2,2011-01-01 01:00:00,7545.29,Bankstown,19.5,8694.851852,7257.41,7161.66,6870.62,7483.69
3,2011-01-01 01:30:00,7206.58,Bankstown,19.1,8720.783208,6889.21,6842.79,6557.80,7117.23
4,2011-01-01 02:00:00,6824.75,Bankstown,18.6,8754.530000,6608.99,6594.82,6237.94,6812.03


## Section 11. Add Other Time-Based Features Discovered in EDA

Weekday/Weekend, Hour, Month

In [ ]:
# df_model['HOUR'] = df_demand['DATETIME'].dt.hour
# df_model['DAYOFWEEK'] = df_demand['DATETIME'].dt.dayofweek
# df_model['IS_WEEKEND'] = df_model['DAYOFWEEK'].apply(lambda x: 1 if x >= 5 else 0)
# df_model['MONTH'] = df_demand['DATETIME'].dt.month
# df_model['YEAR'] = df_demand['DATETIME'].dt.year

Covid Dates (Add this to EDA)

In [23]:
#df_model['IS_COVID'] = np.where((df_model["DATETIME"]>=pd.Timestamp("2020-01-01")) & (df_model["DATETIME"]<=pd.Timestamp("2022-12-31")), 1, 0)

In [163]:
df_model.head(10)

,DATETIME,TOTALDEMAND,FORECASTDEMAND,BEST_FORECASTDEMAND,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago,TEMPERATURE
0,2011-01-01 00:00:00,7953.990,8667.620562,7740.500,7506.450,7249.775,7923.655,20.650000
1,2011-01-01 01:00:00,7375.935,8707.817530,7073.310,7002.225,6714.210,7300.460,19.300000
2,2011-01-01 02:00:00,6696.450,8774.585784,6494.700,6500.915,6131.130,6678.180,18.616667
3,2011-01-01 03:00:00,6361.705,8870.708335,6232.990,6302.170,5829.190,6330.085,18.200000
4,2011-01-01 04:00:00,6228.680,8980.747790,6169.175,6342.095,5770.095,6229.900,18.175000
5,2011-01-01 05:00:00,6155.505,9098.295973,6256.185,6603.260,5829.660,6217.980,18.625000
6,2011-01-01 06:00:00,6313.140,9220.453644,6615.875,7128.295,6199.785,6438.450,20.800000
7,2011-01-01 07:00:00,6862.090,9321.507390,7207.435,7864.215,6781.550,6859.580,23.600000
8,2011-01-01 08:00:00,7604.625,9403.481686,7966.205,8543.430,7310.110,7364.465,26.450000
9,2011-01-01 09:00:00,8265.695,9468.810683,8822.075,9229.955,7581.570,7838.500,29.250000


In [162]:
df_model['DATETIME'] = df_model["DATETIME"].dt.floor("1h")
df_model = df_model.groupby(["DATETIME"]).agg({
    "TOTALDEMAND":"mean",
    "FORECASTDEMAND":"mean",
    "BEST_FORECASTDEMAND":"mean",
    "demand_1_day_ago":"mean",
    "demand_1_week_ago":"mean",
    "demand_1_year_ago":"mean",
    "TEMPERATURE":"mean",}).reset_index()

## Section 12. Train / Validation / Test Split

For time-series forecasting, use chronological split.

In [164]:
n = len(df_model)

train_size = int(n * 0.6)
val_size = int(n * 0.2)

train       = df_model.iloc[:train_size]
validation  = df_model.iloc[train_size:train_size + val_size]
test        = df_model.iloc[train_size + val_size:]

## Section 12. Feature Scaling

In [165]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

features = ["TOTALDEMAND",
            "demand_1_day_ago",
            "demand_1_week_ago",
            "demand_1_year_ago",
            "TEMPERATURE", "FORECASTDEMAND","BEST_FORECASTDEMAND"]

train_scaled = scaler.fit_transform(train[features])
val_scaled = scaler.transform(validation[features])
test_scaled = scaler.transform(test[features])


Feature scaling is not required for statistical models such as SARIMAX, as these models operate directly on the original scale of the data. However, scaling may be applied when training neural network-based models such as PatchTST to improve training stability and convergence.

## Section 13. Saving Preprocessed Datasets for Modelling

After completing the preprocessing and feature engineering steps, the final datasets are exported as CSV files for reuse in the modelling stage. This step ensures that the preprocessing pipeline does not need to be recomputed repeatedly when training different models. The processed dataset (df_model) and its corresponding train, validation, and test splits are saved separately. In addition, scaled versions of the datasets are also stored for use in deep learning models such as LSTM and PatchTST, which require normalized input features for stable training.

Saving these datasets improves workflow reproducibility and modularity. Subsequent modelling notebooks can directly load the processed data without rerunning the entire preprocessing procedure. This approach also reduces computational overhead and helps maintain consistency across different model experiments.

The following datasets are saved:

- df_model.csv: Final dataset after preprocessing and feature engineering

- train.csv: Training dataset

- validation.csv: Validation dataset

- test.csv: Test dataset

- train_scaled.csv: Scaled training dataset for neural network models

- val_scaled.csv: Scaled validation dataset

- test_scaled.csv: Scaled test dataset

These files are stored in the directory:

capstone_project_GroupA/data/NSW/

This structure allows different modelling notebooks (train.csv, test.csv and validation.csv for SARIMAX and train_scaled, test_scaled and val_scaled for LSTM, PatchTST) to load the same consistent datasets.

In [169]:
# Convert scaled numpy arrays into DataFrames so they can be saved as CSV files
# This preserves column names and datetime index for later modelling

features = [
    "TOTALDEMAND",
    "TEMPERATURE",
    "FORECASTDEMAND",
    "BEST_FORECASTDEMAND",
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
]

train_scaled = pd.DataFrame(train_scaled, columns=features, index=train.index)
val_scaled = pd.DataFrame(val_scaled, columns=features, index=validation.index)
test_scaled = pd.DataFrame(test_scaled, columns=features, index=test.index)


# ---------------------------------------------------------
# Save all processed datasets for use in modelling notebooks
# ---------------------------------------------------------

import os

repo_path = "capstone_project_GroupA"
output_path = os.path.join(repo_path, "data", "NSW")

# Create folder if it does not exist
os.makedirs(output_path, exist_ok=True)

# Save main modelling dataset
df_model.to_csv(os.path.join(output_path, "df_model.csv"))

# Save train / validation / test splits
train.to_csv(os.path.join(output_path, "train.csv"))
validation.to_csv(os.path.join(output_path, "validation.csv"))
test.to_csv(os.path.join(output_path, "test.csv"))

# Save scaled datasets (used for LSTM and PatchTST models)
train_scaled.to_csv(os.path.join(output_path, "train_scaled.csv"))
val_scaled.to_csv(os.path.join(output_path, "val_scaled.csv"))
test_scaled.to_csv(os.path.join(output_path, "test_scaled.csv"))

# Display confirmation
print("Saved files to:", output_path)
print(os.listdir(output_path))

Saved files to: capstone_project_GroupA\data\NSW
['df_model.csv', 'forecastdemand_nsw.csv.zip', 'forecastdemand_nsw.csv.zip.partaa', 'forecastdemand_nsw.csv.zip.partab', 'temperature_nsw.csv.zip', 'test.csv', 'test_scaled.csv', 'totaldemand_nsw.csv.zip', 'train.csv', 'train_scaled.csv', 'validation.csv', 'val_scaled.csv']
